In [1]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F

In [2]:
# Toy vocabulary and sentence
vocab_size = 20
embedding_dim = 16
num_heads = 2
hidden_dim = 32
max_len = 5  # generate up to 5 tokens

In [3]:
# 1️⃣ Token embedding
token_embedding = nn.Embedding(vocab_size, embedding_dim)

In [4]:
# 2️⃣ Positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)
    def forward(self, x):
        return x + self.pe[:x.size(0), :]

pos_encoder = PositionalEncoding(embedding_dim)

In [5]:
# 3️⃣ Decoder-only layer
decoder_layer = nn.TransformerDecoderLayer(d_model=embedding_dim, nhead=num_heads, dim_feedforward=hidden_dim)
transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=1)

In [6]:
# 4️⃣ Autoregressive generation loop
generated_tokens = [torch.tensor([0])]  # start token, shape [1]

for step in range(max_len):
    # Prepare input sequence
    input_ids = torch.cat(generated_tokens)  # shape [seq_len]
    token_emb = token_embedding(input_ids)   # [seq_len, embedding_dim]
    token_emb = pos_encoder(token_emb)
    token_emb = token_emb.unsqueeze(1)       # [seq_len, batch_size=1, embedding_dim]

    # Causal mask
    seq_len = token_emb.size(0)
    tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len)

    # Forward pass (memory is empty for decoder-only)
    memory = torch.zeros(0, 1, embedding_dim)
    output = transformer_decoder(tgt=token_emb, memory=memory, tgt_mask=tgt_mask)

    # Map last token embedding to logits (toy linear layer)
    linear = nn.Linear(embedding_dim, vocab_size)
    logits = linear(output[-1, 0])
    probs = F.softmax(logits, dim=-1)

    # Sample next token
    next_token = torch.multinomial(probs, num_samples=1)  # shape [1]
    generated_tokens.append(next_token)

# Convert to indices
generated_indices = [t.item() for t in generated_tokens]
print("Generated token indices:", generated_indices)

Generated token indices: [0, 0, 1, 11, 14, 17]
